# WorldSim v2 GGUF Conversion
Merge LoRA adapter -> HF checkpoint -> GGUF -> Q4_K_M quantization

## 1. Environment & Adapter Discovery

In [1]:
from pathlib import Path
import os
import sys

cwd = Path.cwd()
repo_marker = cwd / 'training/run_qlora_train.py'
notebook_marker = cwd / 'notebooks/dgx_spark_gguf_conversion.ipynb'
assert repo_marker.exists() and notebook_marker.exists(), (
    'Run this notebook from the repository root: worldsim-training'
)
REPO_ROOT = cwd
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from training.lib.qlora_smoke import BASELINE_DATASET_ID, BASELINE_MODEL_NAME

GGUF_OUTPUT_DIR = REPO_ROOT / 'artifacts' / 'gguf'
GGUF_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

outputs_dir = REPO_ROOT / 'outputs' / 'baseline' / BASELINE_DATASET_ID
if not outputs_dir.exists():
    raise FileNotFoundError(f'No training outputs found at {outputs_dir}')

runs = sorted(
    [path for path in outputs_dir.iterdir() if path.is_dir() and path.name.startswith('run-')],
    reverse=True,
)
if not runs:
    raise FileNotFoundError(f'No run directories found under {outputs_dir}')

latest_run = runs[0]
adapter_dir = latest_run / 'adapter'
if not adapter_dir.exists():
    raise FileNotFoundError(f'No adapter found at {adapter_dir}')

print(f'Repo root:    {REPO_ROOT}')
print(f'Base model:   {BASELINE_MODEL_NAME}')
print(f'Dataset id:   {BASELINE_DATASET_ID}')
print(f'Latest run:   {latest_run.name}')
print(f'Adapter dir:  {adapter_dir}')
print(f'GGUF output:  {GGUF_OUTPUT_DIR}')

adapter_files = sorted(path.name for path in adapter_dir.iterdir())
print(f'\nAdapter files: {adapter_files}')
assert any('adapter' in name for name in adapter_files), 'No adapter files found!'
print('Adapter found: ✅')


Repo root:    /home/hyunlord/github/worldsim-training
Base model:   Qwen/Qwen3.5-0.8B-Base
Dataset id:   worldsim-v31-mix-v1
Latest run:   run-20260314T173504Z
Adapter dir:  /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v31-mix-v1/run-20260314T173504Z/adapter
GGUF output:  /home/hyunlord/github/worldsim-training/artifacts/gguf

Adapter files: ['README.md', 'adapter_config.json', 'adapter_model.safetensors', 'chat_template.jinja', 'tokenizer.json', 'tokenizer_config.json', 'training_args.bin']
Adapter found: ✅


## 2. LoRA Merge

In [2]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

MERGED_DIR = latest_run / 'merged_bf16'

if MERGED_DIR.exists() and (MERGED_DIR / 'config.json').exists():
    print(f'Already merged at {MERGED_DIR}, skipping... ✅')
else:
    print('Loading base model in BF16 (no quantization for merge)...')
    base_model = AutoModelForCausalLM.from_pretrained(
        BASELINE_MODEL_NAME,
        torch_dtype=torch.bfloat16,
        device_map='auto',
        trust_remote_code=False,
    )
    tokenizer = AutoTokenizer.from_pretrained(BASELINE_MODEL_NAME, trust_remote_code=False)

    print('Loading LoRA adapter...')
    model = PeftModel.from_pretrained(base_model, str(adapter_dir))

    print('Merging LoRA weights into base model...')
    merged_model = model.merge_and_unload()

    print(f'Saving merged model to {MERGED_DIR}...')
    MERGED_DIR.mkdir(parents=True, exist_ok=True)
    merged_model.save_pretrained(str(MERGED_DIR))
    tokenizer.save_pretrained(str(MERGED_DIR))

    del merged_model, model, base_model
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    merged_size_gb = sum(path.stat().st_size for path in MERGED_DIR.rglob('*') if path.is_file()) / 1e9
    print(f'Merged model saved: {merged_size_gb:.2f} GB ✅')

/home/hyunlord/github/diffusion-study/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Already merged at /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v31-mix-v1/run-20260314T173504Z/merged_bf16, skipping... ✅


## 3. Build llama.cpp (if needed)

In [3]:
import subprocess

LLAMA_CPP_DIR = REPO_ROOT / 'tools' / 'llama.cpp'

if (LLAMA_CPP_DIR / 'build' / 'bin' / 'llama-quantize').exists():
    print(f'llama.cpp already built at {LLAMA_CPP_DIR} ✅')
else:
    print('Building llama.cpp from source...')
    if not (LLAMA_CPP_DIR / 'CMakeLists.txt').exists():
        LLAMA_CPP_DIR.parent.mkdir(parents=True, exist_ok=True)
        subprocess.run(
            ['git', 'clone', '--depth=1', 'https://github.com/ggml-org/llama.cpp.git', str(LLAMA_CPP_DIR)],
            check=True,
        )

    build_dir = LLAMA_CPP_DIR / 'build'
    build_dir.mkdir(exist_ok=True)
    subprocess.run(
        ['cmake', '..', '-DGGML_CUDA=ON', '-DCMAKE_BUILD_TYPE=Release'],
        cwd=str(build_dir),
        check=True,
    )
    subprocess.run(
        ['cmake', '--build', '.', '--config', 'Release', '-j', str(os.cpu_count() or 4)],
        cwd=str(build_dir),
        check=True,
    )

    for binary in ['llama-quantize', 'llama-cli', 'llama-server', 'llama-completion']:
        binary_path = build_dir / 'bin' / binary
        assert binary_path.exists(), f'Build failed: {binary} not found at {binary_path}'

    print('llama.cpp built successfully ✅')

LLAMA_QUANTIZE = LLAMA_CPP_DIR / 'build' / 'bin' / 'llama-quantize'
LLAMA_CLI = LLAMA_CPP_DIR / 'build' / 'bin' / 'llama-cli'
LLAMA_COMPLETION = LLAMA_CPP_DIR / 'build' / 'bin' / 'llama-completion'
LLAMA_SERVER = LLAMA_CPP_DIR / 'build' / 'bin' / 'llama-server'
CONVERT_SCRIPT = LLAMA_CPP_DIR / 'convert_hf_to_gguf.py'
print(f'llama-quantize:    {LLAMA_QUANTIZE}')
print(f'llama-cli:         {LLAMA_CLI}')
print(f'llama-completion:  {LLAMA_COMPLETION}')
print(f'llama-server:      {LLAMA_SERVER}')
print(f'convert script:    {CONVERT_SCRIPT}')


llama.cpp already built at /home/hyunlord/github/worldsim-training/tools/llama.cpp ✅
llama-quantize:    /home/hyunlord/github/worldsim-training/tools/llama.cpp/build/bin/llama-quantize
llama-cli:         /home/hyunlord/github/worldsim-training/tools/llama.cpp/build/bin/llama-cli
llama-completion:  /home/hyunlord/github/worldsim-training/tools/llama.cpp/build/bin/llama-completion
llama-server:      /home/hyunlord/github/worldsim-training/tools/llama.cpp/build/bin/llama-server
convert script:    /home/hyunlord/github/worldsim-training/tools/llama.cpp/convert_hf_to_gguf.py


## 4. Convert HF -> GGUF (FP16)

In [4]:
FP16_GGUF = latest_run / 'worldsim-v2-qwen3.5-0.8b-f16.gguf'

if FP16_GGUF.exists():
    print(f'FP16 GGUF already exists: {FP16_GGUF} ({FP16_GGUF.stat().st_size / 1e9:.2f} GB) ✅')
else:
    print('Converting HF -> GGUF (FP16)...')
    result = subprocess.run(
        [
            sys.executable,
            str(CONVERT_SCRIPT),
            str(MERGED_DIR),
            '--outfile',
            str(FP16_GGUF),
            '--outtype',
            'f16',
        ],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f'STDERR: {result.stderr[-2000:]}')
        raise RuntimeError('GGUF conversion failed')

    print(f'FP16 GGUF created: {FP16_GGUF.stat().st_size / 1e9:.2f} GB ✅')

FP16 GGUF already exists: /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v31-mix-v1/run-20260314T173504Z/worldsim-v2-qwen3.5-0.8b-f16.gguf (1.52 GB) ✅


## 5. Quantize -> Q4_K_M

In [5]:
Q4_GGUF = latest_run / 'worldsim-v2-qwen3.5-0.8b-q4_k_m.gguf'

if Q4_GGUF.exists():
    print(f'Q4_K_M GGUF already exists: {Q4_GGUF} ({Q4_GGUF.stat().st_size / 1e6:.0f} MB) ✅')
else:
    print('Quantizing FP16 -> Q4_K_M...')
    result = subprocess.run(
        [str(LLAMA_QUANTIZE), str(FP16_GGUF), str(Q4_GGUF), 'Q4_K_M'],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f'STDERR: {result.stderr[-2000:]}')
        raise RuntimeError('Quantization failed')

    print(f'Q4_K_M GGUF created: {Q4_GGUF.stat().st_size / 1e6:.0f} MB ✅')

Q4_K_M GGUF already exists: /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v31-mix-v1/run-20260314T173504Z/worldsim-v2-qwen3.5-0.8b-q4_k_m.gguf (529 MB) ✅


## 6. Verify GGUF

In [6]:
import json
import re

print("=== GGUF Verification ===")
print(f"File: {Q4_GGUF}")
print(f"Size: {Q4_GGUF.stat().st_size / 1e6:.0f} MB")

# Use llama-completion instead of llama-cli to avoid interactive mode hang.
# llama-cli enters a conversation loop after generation, causing subprocess
# timeout. llama-completion runs one-shot and exits cleanly.

def strip_think_tags(text: str) -> str:
    """Remove <think>...</think> wrapper from model output."""
    return re.sub(r'<think>.*?</think>\s*', '', text, flags=re.DOTALL).strip()

print("\n--- Test 1: Basic generation (with grammar) ---")
test_prompt = """[TASK] E
[TEMP]
NS=0.8 HA=0.2 RD=0.5 P=0.7 type=choleric
[인물 성격]
당당함, 충동적
[벌어진 일]
낯선 무리가 다가온다
[선택지]
0:도망 1:숨기 2:맞서기 3:경고
[출력 형식]
{"action_id": 번호, "confidence": 0.0, "hint_ko": "한 문장", "hint_en": "one sentence", "personality_reasoning": "축", "temperament_factor": "phrase"}"""

grammar = """root ::= "{" ws "\"action_id\":" ws number "," ws "\"confidence\":" ws float "," ws "\"hint_ko\":" ws string "," ws "\"hint_en\":" ws string "," ws "\"personality_reasoning\":" ws string "," ws "\"temperament_factor\":" ws string ws "}"
ws ::= [ \t\n]*
number ::= [0-3]
float ::= "0." [0-9] [0-9]?
string ::= "\"" [^"\\]* "\""
"""

grammar_file = latest_run / 'test_grammar.gbnf'
grammar_file.write_text(grammar, encoding='utf-8')

result = subprocess.run(
    [
        str(LLAMA_COMPLETION),
        '-m', str(Q4_GGUF),
        '-p', test_prompt,
        '-n', '128',
        '--grammar-file', str(grammar_file),
        '--temp', '0',
        '-t', '4',
        '-ngl', '99',
        '--no-display-prompt',
    ],
    capture_output=True,
    text=True,
    timeout=60,
)
output_text = strip_think_tags(result.stdout.strip())
print(f"Output: {output_text[:500]}")

try:
    parsed = json.loads(output_text)
    print(f"Valid JSON: action_id={parsed.get('action_id')}, confidence={parsed.get('confidence')} ✅")
except json.JSONDecodeError as exc:
    print(f"JSON parse failed: {exc}")
    print(f"Raw output: {output_text[:200]}")

print("\n--- Test 2: Korean narrative ---")
test_prompt_ko = """[TASK] B
[TEMP]
NS=0.2 HA=0.8 RD=0.6 P=0.4 type=melancholic
[인물 성격]
신중함, 불안함
[지금 느끼는 것]
두려움:0.8
[벌어진 일]
짐승발견"""

result_ko = subprocess.run(
    [
        str(LLAMA_COMPLETION),
        '-m', str(Q4_GGUF),
        '-p', test_prompt_ko,
        '-n', '128',
        '--temp', '0.7',
        '-t', '4',
        '-ngl', '99',
        '--no-display-prompt',
    ],
    capture_output=True,
    text=True,
    timeout=60,
)
output_ko = strip_think_tags(result_ko.stdout.strip())
print(f"Output: {output_ko[:500]}")


=== GGUF Verification ===
File: /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v31-mix-v1/run-20260314T173504Z/worldsim-v2-qwen3.5-0.8b-q4_k_m.gguf
Size: 529 MB

--- Test 1: Basic generation (with grammar) ---
Output: {"action_id": 2, "confidence": 0.7, "hint_ko": "낯선 무리가 다가오면 먼저 숨어 지켜보자", "hint_en": "dangerous stranger: stay hidden to watch carefully", "personality_reasoning": "당당함", "temperament_factor": "choleric"}

> EOF by user
JSON parse failed: Extra data: line 3 column 1 (char 205)
Raw output: {"action_id": 2, "confidence": 0.7, "hint_ko": "낯선 무리가 다가오면 먼저 숨어 지켜보자", "hint_en": "dangerous stranger: stay hidden to watch carefully", "personality_reasoning": "당당함", "temperament_factor": "choleri

--- Test 2: Korean narrative ---
Output: [벌어진 일]
짐승 발견, 짐승의 잔물길

[지금 느끼는 것]
두려움:0.8

> EOF by user


## 7. Copy to artifacts/gguf

In [7]:
import shutil

final_gguf = GGUF_OUTPUT_DIR / 'worldsim-v2-qwen3.5-0.8b-q4_k_m.gguf'
shutil.copy2(Q4_GGUF, final_gguf)

print(f'Final GGUF: {final_gguf} ✅')
print(f'   Size: {final_gguf.stat().st_size / 1e6:.0f} MB')
print('   Ready for game distribution')

print('\n' + '=' * 60)
print('GGUF CONVERSION COMPLETE')
print('=' * 60)
print(f'  Base model:     {BASELINE_MODEL_NAME}')
print(f'  Adapter:        {adapter_dir}')
print(f'  Merged BF16:    {MERGED_DIR}')
print(f'  FP16 GGUF:      {FP16_GGUF} ({FP16_GGUF.stat().st_size / 1e6:.0f} MB)')
print(f'  Q4_K_M GGUF:    {final_gguf} ({final_gguf.stat().st_size / 1e6:.0f} MB)')
print('  Quantization:   Q4_K_M')
print('=' * 60)


Final GGUF: /home/hyunlord/github/worldsim-training/artifacts/gguf/worldsim-v2-qwen3.5-0.8b-q4_k_m.gguf ✅
   Size: 529 MB
   Ready for game distribution

GGUF CONVERSION COMPLETE
  Base model:     Qwen/Qwen3.5-0.8B-Base
  Adapter:        /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v31-mix-v1/run-20260314T173504Z/adapter
  Merged BF16:    /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v31-mix-v1/run-20260314T173504Z/merged_bf16
  FP16 GGUF:      /home/hyunlord/github/worldsim-training/outputs/baseline/worldsim-v31-mix-v1/run-20260314T173504Z/worldsim-v2-qwen3.5-0.8b-f16.gguf (1517 MB)
  Q4_K_M GGUF:    /home/hyunlord/github/worldsim-training/artifacts/gguf/worldsim-v2-qwen3.5-0.8b-q4_k_m.gguf (529 MB)
  Quantization:   Q4_K_M
